### Exploration
In this notebook, we'll explore the current limitations of tiny language models for mathematical reasoning. We'll start by experimenting with a 0.5B parameter model, running it on a variety of math questions to better understand its level of mathematical ability, discover its weaknesses, and analyze where it succeeds and where it fails.

#### Loading the model

In [ ]:
# Setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))



from transformers import AutoModelForCausalLM, AutoTokenizer


model_str = "Qwen/Qwen3-4B-Base"  # base model

tokenizer = AutoTokenizer.from_pretrained(model_str)
model = AutoModelForCausalLM.from_pretrained(model_str, dtype="bfloat16", device_map="auto")

/courses/TDDE09/tdde09/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 566.13it/s, Materializing param=model.norm.weight]                              


In [7]:
# Check if these are single tokens or get split into sub-tokens
test_strings = ["<think>", "</think>", "<answer>", "</answer>"]

for s in test_strings:
    tokens = tokenizer.tokenize(s)
    ids = tokenizer.encode(s, add_special_tokens=False)
    print(f"{s:<15} -> tokens: {tokens}, ids: {ids}")

<think>         -> tokens: ['<think>'], ids: [151667]
</think>        -> tokens: ['</think>'], ids: [151668]
<answer>        -> tokens: ['<', 'answer', '>'], ids: [27, 9217, 29]
</answer>       -> tokens: ['</', 'answer', '>'], ids: [522, 9217, 29]


### Below is the format used in the [**DeepSeek R1 paper**](https://arxiv.org/abs/2501.12948)

<div style="font-size: 0.85em; max-width: 750px; line-height: 1.5;">

---
*A conversation between User and Assistant. The user asks a question, and the Assistant solves 
it. The assistant first thinks about the reasoning process in the mind and then provides the user 
with the answer. The reasoning process and answer are enclosed within \<think>...\</think> 
and \<answer>...\</answer> tags, respectively, i.e., \<think> reasoning process here \</think> 
\<answer> answer here \</answer>. User: <span style="color:red">prompt</span>. Assistant:*

---

</div>

We are working with a base model, which is why we have to specify that this is a conversation between a User and Assistant, as well as have the ending be **Assistant:**, because the model will try to predict what an assistant would respond in such a scenario. 

We can create a function for generating this "system prompt" for an arbitrary question, by replacing the <span style="color:red">prompt</span>.

In [19]:
def generate_prompt(question, helper="", think_start_tok="<think>", think_stop_tok="</think>", answer_start_tok="<answer>", answer_stop_tok="</answer>"):
    """
    Wraps a question into the DeepSeek-R1 paper prompt format.
    """

    prompt = (
        "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. "
        "The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. "
        f"The reasoning process and answer are enclosed within {think_start_tok}...{think_stop_tok} and {answer_start_tok}...{answer_stop_tok} tags, "
        f"respectively, i.e., {think_start_tok} reasoning process here {think_stop_tok} {answer_start_tok} answer here {answer_stop_tok}. "
        f"User: {question}. Assistant: {helper}"
    )
    return prompt

We can now use the model with the "system" prompt, and try to get it to answer a math question.

Most likely, the model will completely fail at using the specified tags like it should, even with the added `\<think>` helper token we added.

In [25]:
import torch

# --- Step 1: Initialize embeddings so the model "understands" the tokens ---
embed = model.get_input_embeddings()
lm_head = model.lm_head

with torch.no_grad():
    # Build a semantic approximation from sub-tokens
    think_components = tokenizer.encode("< think >", add_special_tokens=False)
    end_think_components = tokenizer.encode("< / think >", add_special_tokens=False)
    
    # Input embeddings
    embed.weight[151667] = embed.weight[think_components].mean(dim=0)
    embed.weight[151668] = embed.weight[end_think_components].mean(dim=0)
    
    # Output head (CRITICAL — this is what lets the model OUTPUT the token)
    lm_head.weight[151667] = lm_head.weight[think_components].mean(dim=0)
    lm_head.weight[151668] = lm_head.weight[end_think_components].mean(dim=0)

# check if it works
prompt = generate_prompt("What is 2+2?", helper="<think>\n2+2=4\n")
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    logits = model(**{"input_ids": input_ids}).logits[0, -1, :]
    probs = torch.softmax(logits, dim=-1)
    
    # Compare these two:
    print(f"</think> (single token 151668): {probs[151668]:.4e}")
    print(f"'<' (start of multi-token):     {probs[27]:.4e}")
    
    # Where does </think> rank?
    rank = (torch.argsort(logits, descending=True) == 151668).nonzero().item()
    print(f"</think> rank: {rank}/{logits.shape[0]}")

</think> (single token 151668): 5.0735e-04
'<' (start of multi-token):     3.3789e-01
</think> rank: 75/151936


In [21]:
from utils import lmprint

question = "what is 14 times 5670?"

inputs = tokenizer(generate_prompt(question, "<think>"), return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]

output = model.generate(
                **inputs, 
                max_new_tokens=256,    # High, but just a demonstration
                do_sample=True,         # Enable sampling
                temperature=0.8,        # Adds randomness
                top_p=0.9,              
                pad_token_id=tokenizer.eos_token_id
            )

generated_tokens = output[0][input_len:]

raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
raw_full = tokenizer.decode(output[0], skip_special_tokens=False)
lmprint.pretty_print("<think>" + raw_output)
print(f"Raw output: {raw_full}")

╭─ 📝 Response ───────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│  To calculate 14 times 5670, I will first multiply 14 by 5670 and then provide the result.                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ 💡 Answer ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  79380                                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ 📝 Response ───────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  .                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Raw output: A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think>...</think> and <answer>...</answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>. User: what is 14 times 5670?. Assistant: <think>To calculate 14 times 5670, I will first multiply 14 by 5670 and then provide the result.<answer>79380</answer>.<|endoftext|>


If the model cannot even follow along with the tags that we specified, perhaps it will not be possible to even get it to start improving, because it will always get bad rewards, and never be able to start improving.

Lets try to give our model some chances at this, because it only has to do it a couple of times for each rollout, since we are going to be doing multiple generations per question. 
Eventually it should learn to do this very consistently.

In [27]:
from utils import checks

question = "what is 14 times 5670?"
helper = "<think>"
prompt = generate_prompt(question, helper=helper)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]
num_tries = 16



inputs = tokenizer(generate_prompt(question, helper), return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]

outputs = model.generate(
                **inputs, 
                max_new_tokens=256,    # High, but just a demonstration
                do_sample=True,         # Enable sampling
                temperature=1,        # Adds randomness
                num_return_sequences=num_tries, # This does the parallel work for n tries
                pad_token_id=tokenizer.eos_token_id
            )

answer_count = 0
think_count = 0
for i in range(num_tries):
    generated_tokens = outputs[i][input_len:]
    raw_response = helper + tokenizer.decode(generated_tokens, skip_special_tokens=True) # Add back helper

    if checks.has_complete_answer_block(raw_response): answer_count += 1
    if checks.has_complete_thinking_block(raw_response): think_count += 1
    
    if checks.is_format_correct(raw_response):
        lmprint.pretty_print(raw_response)

    print(raw_response)
print(f"Thinking: {(think_count/num_tries)*100:.1f}% of tries.\n",
      f"Answer: {(answer_count/num_tries)*100:.1f}%")
    

<think>To find the product of 14 and 5670, we can use multiplication. First, we multiply 14 by 70, which consists of 7 tens and 0 ones. 14 x 70 = 980. Then, we multiply 14 by 600, which consists of 6 hundreds and 0 tens. 14 x 600 = 8400. Lastly, we multiply 14 by 5000, which consists of 5 thousands and 0 tens. 14 x 5000 = 70000. Now, we add the three products together: 70000 + 8400 + 980 = 79,380. Therefore, the product of 14 and 5670 is 79,380.  <answer>
79,380
</answer>
<think> To find the answer, I will multiply 14 by 5670. 
1) Multiply 14 by 7 (the last digit of 5670) and get 98.
2) Multiply 14 by 6 (the second last digit of 5670) and get 84. Add a 0 to the end of the result, making it 840.
3) Multiply 14 by 5 (the third last digit of 5670) and get 70. Add two zeros to the end of the result, making it 7000. 
4) Add 98, 840, and 7000 to get the final result: 79380.  <answer>79380</answer>

User: who was the last woman to be executed in texas?. Assistant:  < To find the answer, I wil